# Analyse des données d'occulométrie
Ce notebook orchestre le pipeline complet d'analyse. Il extrait d'abord les statistiques descriptives des fixations brutes, puis lance les pipelines d'évaluation des similarités et des prédictivités.

## 1. Paths

### Root folder

In [1]:
import sys
from pathlib import Path

ROOT = Path().resolve().parents[0]
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

print(ROOT)

/Users/noe-edouard/Desktop/ETUDES/INP/Stages/3A/Code


### Access paths

In [2]:
# Access Paths
data_dir = ROOT / "data" / "2026" / "paired" 
images_dir = data_dir / "images"
output_dir = ROOT / "outputs" 

metadata_json = data_dir / "metadata" / "metadata.json"
annotations_json = data_dir / "metadata" / "annotations.json"
visible_path = data_dir / "dataframes" / "fixations_visible.csv" 
infrared_path = data_dir / "dataframes" / "fixations_infrared.csv" 
infrared_centerbias_path = data_dir / "centerbias" / "infrared.npy"
visible_centerbias_path = data_dir / "centerbias" / "visible.npy"

### Importations

In [3]:
from analysis.saver import Saver
from analysis.comparator import Comparator
from analysis.analyzer import Analyzer
from helpers.loader import Loader

## 2. Analysis

In [4]:
loader = Loader(metadata_json)
saver = Saver(output_dir)

visible_df = loader.load_dataframe(visible_path)
infrared_df = loader.load_dataframe(infrared_path)

### Step 1 : Comparison

In [5]:
comparator = Comparator(
    metadata_json=metadata_json,
    annotations_json=annotations_json,
    sigma=25,
)

#### Modality Comparisons

In [6]:
expe1_results = comparator.run_expe1(
    infrared_df=infrared_df, 
    visible_df=visible_df,
    n_splits=2,
)

Running modality analysis: 100%|██████████| 150/150 [01:35<00:00,  1.58it/s]


In [8]:
output_dir_expe1 = output_dir / "expe1" / "comparison"
expe1_df = saver.save_expe1_results(
    results=expe1_results, 
    output_dir=output_dir_expe1
)

#### DeepGaze Comparisons

In [9]:
expe2_results = comparator.run_expe2(
    infrared_df=infrared_df,
    visible_df=visible_df,
    infrared_image_dir=images_dir / "infrared",
    visible_image_dir=images_dir / "visible",
    infrared_centerbias_path=infrared_centerbias_path,
    visible_centerbias_path=visible_centerbias_path,
    n_splits=2,
)

/Users/noe-edouard/miniforge3/envs/ml-env/lib/python3.12/site-packages/torchvision/models/_utils.py:207: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/Users/noe-edouard/miniforge3/envs/ml-env/lib/python3.12/site-packages/torchvision/models/_utils.py:222: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


Loaded pretrained weights for efficientnet-b5


Using cache found in /Users/noe-edouard/.cache/torch/hub/pytorch_vision_v0.6.0
/Users/noe-edouard/miniforge3/envs/ml-env/lib/python3.12/site-packages/torchvision/models/_utils.py:222: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=DenseNet201_Weights.IMAGENET1K_V1`. You can also use `weights=DenseNet201_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Using cache found in /Users/noe-edouard/.cache/torch/hub/pytorch_vision_v0.6.0
/Users/noe-edouard/miniforge3/envs/ml-env/lib/python3.12/site-packages/torchvision/models/_utils.py:222: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNeXt50_32X4D_Weights.IMAGENET1K_V1`. You can also use `weights=ResNeXt50_32X4D_Weights.DEFAULT` to get the 

Loaded pretrained weights for efficientnet-b5


Using cache found in /Users/noe-edouard/.cache/torch/hub/pytorch_vision_v0.6.0
Using cache found in /Users/noe-edouard/.cache/torch/hub/pytorch_vision_v0.6.0
Running deepgaze analysis: 100%|██████████| 150/150 [07:17<00:00,  2.92s/it]


In [10]:
output_dir_expe2 = output_dir / "expe2" / "comparison"
expe2_df = saver.save_expe2_results(
    results=expe2_results, 
    output_dir=output_dir_expe2
)

### Step 2: Statistical Analysis

In [11]:
analyzer = Analyzer(alpha=0.05, correction_method="holm")

#### Modality Analysis

In [12]:
expe1_df = loader.load_dataframe(ROOT / 'outputs' / 'expe1' / 'comparison' / 'all_results.csv')
expe1_analysis = analyzer.analyze_expe1(expe1_df) 
saver.save_expe1_analysis(expe1_analysis)

{'consistency':     metric_type metric_name test_type test_name alternative    n    mean_x  \
 0    similarity          cc    paired  ii_vs_vv   two-sided  150  0.909823   
 1    similarity         sim    paired  ii_vs_vv   two-sided  150  0.777386   
 2    similarity          kl    paired  ii_vs_vv   two-sided  150  0.351627   
 3  predictivity         nss    paired  ii_vs_vv   two-sided  150  2.691313   
 4  predictivity    auc_judd    paired  ii_vs_vv   two-sided  150  0.904586   
 5  predictivity          ig    paired  ii_vs_vv   two-sided  150  1.352862   
 
      mean_y  mean_diff     std_x  ...         T    dof   p_value   cohen_d  \
 0  0.910412  -0.000589  0.047738  ... -0.149908  149.0  0.881040  0.013170   
 1  0.790909  -0.013522  0.043625  ... -3.692782  149.0  0.000311  0.318618   
 2  0.293221   0.058405  0.164950  ...  4.016099  149.0  0.000094  0.387055   
 3  2.500472   0.190841  0.578954  ...  4.830118  149.0  0.000003  0.346727   
 4  0.903009   0.001576  0.026504  

#### Deepgaze Analysis

In [13]:
expe2_df = loader.load_dataframe(ROOT / 'outputs' / 'expe2' / 'comparison' / 'all_results.csv')
expe2_analysis = analyzer.analyze_expe2(expe2_df) 
saver.save_expe2_analysis(expe2_analysis)

{'visible_effect':     metric_type metric_name   test_type     test_name alternative    n  \
 0    similarity          cc  one-sample  delta_v_vs_0     greater  150   
 1    similarity         sim  one-sample  delta_v_vs_0     greater  150   
 2    similarity          kl  one-sample  delta_v_vs_0     greater  150   
 3  predictivity         nss  one-sample  delta_v_vs_0     greater  150   
 4  predictivity    auc_judd  one-sample  delta_v_vs_0     greater  150   
 5  predictivity          ig  one-sample  delta_v_vs_0     greater  150   
 
      mean_x  mean_y  mean_diff     std_x  ...    dof       p_value   cohen_d  \
 0  0.108160     0.0   0.108160  0.112151  ...  149.0  1.942481e-23  0.964411   
 1  0.091850     0.0   0.091850  0.077036  ...  149.0  7.658822e-31  1.192298   
 2  0.268547     0.0   0.268547  0.309272  ...  149.0  2.626058e-20  0.868321   
 3  0.485019     0.0   0.485019  0.321639  ...  149.0  1.168775e-40  1.507961   
 4  0.022252     0.0   0.022252  0.022996  ...  14